# 01 — Curado de mutaciones COSMIC y descarga de PDBs

**Hito 1 (final día 7).** Este notebook produce:
1. Un CSV unificado de mutaciones somáticas curadas para KRAS, HRAS y NRAS.
2. Un directorio con los PDBs de referencia descargados y validados.

## Contenido
1. Carga del fichero COSMIC en bruto
2. Filtrado: somáticas confirmadas, missense
3. Deduplicación por sample_id
4. Armonización de nomenclatura HGVS
5. Construcción del DataFrame curado
6. Descarga de PDBs y validación
7. Guardado de outputs en `data/processed/`

Lee antes `docs/primeros_pasos.md`, `docs/recursos_tecnicos.md` y `docs/contratos_datos.md`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio.PDB import PDBList, PDBParser

from tfm_ras.config import load_config, project_root
from tfm_ras import cosmic_loader

ROOT = project_root()
CFG = load_config(ROOT / 'configs' / 'config.yaml')
RAW = ROOT / CFG['paths']['data_raw']
EXAMPLE = ROOT / CFG['paths']['data_example']
OUT = ROOT / CFG['paths']['data_processed']
OUT.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG['random_seed'])

## 1. Carga del fichero COSMIC

> **TODO (alumno):** primero usa el TSV de ejemplo para desarrollar. Cuando tengas COSMIC real, colocalo en `data/raw/cosmic/` y documenta la version exacta usada.

In [2]:
real_cosmic = RAW / 'cosmic' / CFG['cosmic']['raw_filename']
example_cosmic = EXAMPLE / CFG['cosmic']['example_filename']
cosmic_path = real_cosmic if real_cosmic.exists() else example_cosmic
print(f'Usando: {cosmic_path.relative_to(ROOT)}')

df_raw = cosmic_loader.load_cosmic_raw(cosmic_path)
df_raw.head()

Usando: data/example/cosmic_minimal_example.tsv


,GENE_NAME,Mutation AA,Mutation somatic status,Mutation Description,ID_sample,Primary site,Histology,COSMIC version
0,KRAS,p.G12D,Confirmed somatic variant,Substitution - Missense,S1,pancreas,adenocarcinoma,example-v1
1,KRAS,p.G12D,Confirmed somatic variant,Substitution - Missense,S1,pancreas,adenocarcinoma,example-v1
2,KRAS,p.G12V,Confirmed somatic variant,Substitution - Missense,S2,large_intestine,adenocarcinoma,example-v1
3,KRAS,p.G13D,Confirmed somatic variant,Substitution - Missense,S3,large_intestine,adenocarcinoma,example-v1
4,KRAS,p.Q61H,Confirmed somatic variant,Substitution - Missense,S4,lung,carcinoma,example-v1


## 2. Filtrado y deduplicación

> **TODO:** aplicar filtros del config y reportar n inicial -> n final por filtro en una tabla. El TSV de ejemplo incluye un duplicado, una mutacion silenciosa y una variante no confirmada para probar los filtros.

In [3]:
df_filtered = cosmic_loader.filter_somatic_missense(df_raw, CFG['cosmic']['filters'])
df_filtered_dedup = cosmic_loader.deduplicate_by_sample(df_filtered)

df_filtered_dedup.head(15)

Conteo Inicial,Mutaciones Somáticas,Mutaciones Missense,Conteo Final
15,14,13,13


,GENE_NAME,Mutation AA,Mutation somatic status,Mutation Description,ID_sample,Primary site,Histology,COSMIC version
0,KRAS,p.G12D,Confirmed somatic variant,Substitution - Missense,S1,pancreas,adenocarcinoma,example-v1
2,KRAS,p.G12V,Confirmed somatic variant,Substitution - Missense,S2,large_intestine,adenocarcinoma,example-v1
3,KRAS,p.G13D,Confirmed somatic variant,Substitution - Missense,S3,large_intestine,adenocarcinoma,example-v1
4,KRAS,p.Q61H,Confirmed somatic variant,Substitution - Missense,S4,lung,carcinoma,example-v1
5,KRAS,p.A146T,Confirmed somatic variant,Substitution - Missense,S5,large_intestine,adenocarcinoma,example-v1
6,HRAS,p.G12V,Confirmed somatic variant,Substitution - Missense,S6,urinary_tract,carcinoma,example-v1
7,HRAS,p.G13R,Confirmed somatic variant,Substitution - Missense,S7,head_and_neck,squamous_cell_carcinoma,example-v1
8,HRAS,p.Q61R,Confirmed somatic variant,Substitution - Missense,S8,thyroid,adenoma,example-v1
9,NRAS,p.G12D,Confirmed somatic variant,Substitution - Missense,S9,haematopoietic_and_lymphoid_tissue,leukaemia,example-v1
10,NRAS,p.G13D,Confirmed somatic variant,Substitution - Missense,S10,haematopoietic_and_lymphoid_tissue,leukaemia,example-v1


## 3. Construccion del DataFrame curado por gen

La salida debe cumplir el contrato de `data/processed/cosmic_curated.csv`. Ver `docs/contratos_datos.md`.

In [4]:
df_curated = cosmic_loader.build_curated_dataset(df_filtered, CFG['cosmic']['version'])
df_curated.head(15)

,gene,uniprot_id,position,wt_aa,mut_aa,hgvs_p,sample_count,tumour_types,primary_tissues,cosmic_version
0,KRAS,P01116,12,G,D,p.G12D,13,adenocarcinoma,pancreas,
1,KRAS,P01116,12,G,D,p.G12D,13,adenocarcinoma,pancreas,
2,KRAS,P01116,12,G,V,p.G12V,13,adenocarcinoma,large_intestine,
3,KRAS,P01116,13,G,D,p.G13D,13,adenocarcinoma,large_intestine,
4,KRAS,P01116,61,Q,H,p.Q61H,13,carcinoma,lung,
5,KRAS,P01116,146,A,T,p.A146T,13,adenocarcinoma,large_intestine,
6,HRAS,P01112,12,G,V,p.G12V,13,carcinoma,urinary_tract,
7,HRAS,P01112,13,G,R,p.G13R,13,squamous_cell_carcinoma,head_and_neck,
8,HRAS,P01112,61,Q,R,p.Q61R,13,adenoma,thyroid,
9,NRAS,P01111,12,G,D,p.G12D,13,leukaemia,haematopoietic_and_lymphoid_tissue,


## 4. Descarga de estructuras PDB

Usa las estructuras principales declaradas en `configs/config.yaml`. Guarda los PDBs en `data/external/pdb/` y valida que Biopython puede parsearlos.

In [5]:
EXTERNAL = ROOT / CFG['paths']['data_external']

output_dir = EXTERNAL
output_dir.mkdir(parents=True, exist_ok=True)

pdbl = PDBList()
parser = PDBParser(QUIET=True)  

for member in CFG['family']['members']:
    ras_gene = member['gene']

    pdb_ids = [
        member['pdb_primary'],
        member['pdb_secondary']
    ]

    for pdb_id in pdb_ids:
        pdb_file = pdbl.retrieve_pdb_file(
            pdb_id, 
            pdir=(output_dir),
            file_format='pdb'
        )

        structure = parser.get_structure(pdb_id, pdb_file)

        print(f'{ras_gene} - {pdb_id}: parse OK')

KRAS - 4OBE: parse OK
KRAS - 5UK9: parse OK
HRAS - 3K8Y: parse OK
HRAS - 2RGE: parse OK
NRAS - 5UHV: parse OK
NRAS - 3CON: parse OK


## 5. Guardado de resultados

El CSV curado se guarda en `data/processed/cosmic_curated.csv`.

Checklist minimo antes de avanzar al notebook 02:

- columnas obligatorias presentes;
- G12/G13/Q61 aparecen si estan en el dataset usado;
- no hay duplicados por `ID_sample` + `Mutation AA`;
- version del dataset registrada.

In [10]:
df_curated.to_csv(OUT/'cosmic_curated.csv', index=False)